# Entrevista

## Data Integration Engineer
- **Candidato**: TBD
- **Stack**: Python · DuckDB · S3 · API Banxico

### Objetivo
Construir desde cero un pipeline ETL productivo que consuma la **API SIE de Banxico**, transforme los datos con **DuckDB** y los persista en la carpeta **s3_datalake** en formato `Parquet`, con plena _observabilidad_ en cada capa.

![img](img/business_case.png)

### Criterios de evaluación
Se valora, en este orden:
1. **Idempotencia** — correr el pipeline dos veces produce el mismo estado final, sin duplicados.
2. **Manejo correcto del caso `UDIMXN.SPOT`** (ver sección dedicada más abajo).
3. **Justificación explícita de DuckDB vs Python** en cada transformación.
4. **Logging** estructurado en cada etapa (API → RAW → curada → estadísticas).
5. **Calidad del SQL y del modelo de datos** (claves naturales, columnas de auditoría, nombres consistentes).

### Notas sobre la API de Banxico
- `fecha` viene como string `DD/MM/YYYY` y `dato` como string — el casting queda en el código del candidato, no en el cliente.
- La metadata de cada serie trae `fechaInicio` y `fechaFin`; esta última **no** siempre coincide con la fecha del último valor *actual* (ver UDIS).

### Security List
Considere la siguiente lista de securities. El `IDSERIE` a usar con la API se resuelve desde la tabla `BANXICO_SERIES` de la base local.

| SECURITY_NAME | Periodicidad efectiva            |
|---------------|----------------------------------|
| `USDMXN.FIX`  | Diaria hábil                     |
| `UDIMXN.SPOT` | Diaria, publicada en bloques (*) |
| `MXN.TPFB`    | Diaria hábil                     |
| `MXN.TIIE-1D` | Diaria hábil                     |

(*) Detalle en la sección **Caso especial UDIMXN.SPOT**.

In [2]:
from vmetrix import get_database

security_list = [
    "USDMXN.FIX",
    "UDIMXN.SPOT",
    "MXN.TPFB",
    "MXN.TIIE-1D"
]

security_list_str = ", ".join(f"'{sec}'" for sec in security_list)

db = get_database()

sql = f"""
select *
from BANXICO_SERIES
where SECURITY_NAME IN ({security_list_str})
"""

db.query(sql)

,SECURITY_NAME,IDSERIE,TITULO,PERIODICIDAD,CIFRA,UNIDAD
0,USDMXN.FIX,SF43718,Tipo de cambio ...,Diaria,Tipo de Cambio,Pesos por Dólar
1,MXN.TIIE-1D,SF331451,"TIIE de Fondeo a Un Día Hábil Bancario, Median...",Diaria,Tasas Promedio,Porcentajes
2,UDIMXN.SPOT,SP68257,Valor de UDIS,Diaria,Tipo de Cambio,Unidades de Inversión
3,MXN.TPFB,SF43773,Tasa de fondeo bancario Mediana ponderada por ...,Diaria,Porcentajes,Sin Unidad


### 1. Carga histórica

Cargue los securities indicados desde **2025-01-01 hasta hoy** en LocalDb.

**Entregables**
1. Documantación y DDL en [vmetrix/database.sql](vmetrix/database.sql) con la(s) definicion(es) de la(s) tabla(s) que diseñe.
2. Creación efectiva de esas tablas en `LocalDb`.
3. Script Python que ejecute la carga.

### 2. Carga diaria incremental

Implemente la arquitectura descrita en la imagen anterior para que el pipeline pueda correr todos los días como un proceso diario.

**2.1 Valores del día**
- Traer los valores vigentes al día de hoy para los cuatro securities.
- Para `UDIMXN.SPOT` ver el caso especial abajo (**NO** se asume un solo valor diario).

**2.2 Estadísticas móviles (7 días)**
Cree una tabla de estadísticas de los últimos **7 días calendario** (ventana `[value_date - 6, value_date]`)
- Evalúe si es necesario una tabla, vista o función para resolver este problema.
    - Justifique su respuesta.

Se usan **7 días calendario** (no hábiles) para que la definición sea la misma para todos los securities

### 3. Requerimientos transversales del ETL

**3.1 RAW en `s3_datalake`**
Almacene cada respuesta de la API como Parquet antes de transformarla. Diseñe la estructura de carpetas bajo [s3_datalake/](s3_datalake/) (simulando buckets S3). 

**3.2 Idempotencia / reprocesabilidad**
- Correr el pipeline dos veces para el mismo rango **NO** debe generar duplicados.
- El proceso debe ser reprocesable.

**3.3 Logging**
- Use el logger `vmetrix` (ya configurado a INFO por defecto) o cree el suyo.
- Implemente el logging con la profundidad y nivel que estime conveniente para cada tarea del pipeline.

**3.4 DuckDB vs Python**
En cada transformación (parseo de fechas, casting de valores, deduplicación, ventana móvil, merges) indique en un comentario corto por qué eligió DuckDB o Python.

---

### Caso especial `UDIMXN.SPOT`

UDIS **sí** tiene un valor **diario** — no son dos datos al mes. Lo que ocurre es que Banxico publica el valor de UDIS **con anticipación en bloques**:

- Alrededor del **día 10** de cada mes se publica el horizonte de valores diarios hasta el **día 25** siguiente.
- Alrededor del **día 25** de cada mes se publica el horizonte hasta el **día 10** del mes siguiente.

Consecuencias prácticas para el pipeline:
1. `get_last_value('SP68257')` devuelve la fecha del **horizonte publicado** (puede estar varios días en el futuro respecto al runtime), *NO* el valor de hoy.
